# **A/B Test Analysis: Impact of Gate Placement on Player Retention (Cookie Cats)**

## Business Problem & Experiment Design

**Context:** Cookie Cats is a mobile puzzle game. Progress is periodically blocked by "gates" that make players wait (or pay) before continuing. The company tested moving the first gate from **level 30** to **level 40**.

**Business question:** Does moving the first gate from level 30 to level 40 improve player retention?

**Groups:** `gate_30` (control, current design) vs `gate_40` (treatment). Players were assigned randomly; each row is one player (90,189 players).

**Metrics**
- Primary: `retention_1` and `retention_7` (player came back 1 / 7 days after install)
- Secondary: `sum_gamerounds` (rounds played in the first 14 days)

**Hypotheses (for each primary metric)**
- H0: Retention rate is the same in both groups (p30 = p40)
- H1: Retention rate differs between the groups (p30 ≠ p40), two-sided

**Decision rule:** alpha = 0.05, adjusted with Bonferroni for the 2 primary tests, so each test uses 0.025.

# **Step 1**

**Step 1.1 - Data Understanding & Data Quality**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chisquare, mannwhitneyu
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep
from statsmodels.stats.power import NormalIndPower

df = pd.read_csv("data/cookie_cats.csv")
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [2]:

df.shape

(90189, 5)

In [3]:

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   userid          90189 non-null  int64 
 1   version         90189 non-null  object
 2   sum_gamerounds  90189 non-null  int64 
 3   retention_1     90189 non-null  bool  
 4   retention_7     90189 non-null  bool  
dtypes: bool(2), int64(2), object(1)
memory usage: 2.2+ MB


In [4]:

# Check missing values in each column
df.isnull().sum()


userid            0
version           0
sum_gamerounds    0
retention_1       0
retention_7       0
dtype: int64

**Step 1.2 - Check for Duplicate Records & User IDs**

In [5]:

# Check fully duplicated rows
df.duplicated().sum()


np.int64(0)

In [6]:

# Check duplicated player IDs
df["userid"].duplicated().sum()


np.int64(0)

**Step 1.3 - Validate Data Values & Business Rules**

In [7]:

# Check unique experiment groups
df["version"].unique()


array(['gate_30', 'gate_40'], dtype=object)

In [8]:

df["version"].value_counts()


version
gate_40    45489
gate_30    44700
Name: count, dtype: int64

In [9]:

# Define valid experiment groups
valid_groups = ["gate_30", "gate_40"]

# Count invalid group values
invalid_groups = ~df["version"].isin(valid_groups)

invalid_groups.sum()


np.int64(0)

In [10]:

# Check negative game rounds
(df["sum_gamerounds"] < 0).sum()


np.int64(0)

In [11]:

# Count players with zero game rounds
(df["sum_gamerounds"] == 0).sum()


np.int64(3994)

**Decision on players with zero game rounds:** All assigned players are kept in the analysis, including 3,994 records with `sum_gamerounds == 0`. Excluding them based on post-assignment activity could bias the group comparison. **Zero rounds do not automatically mean not retained:** 87 of these players have `retention_1 = True` and 29 have `retention_7 = True` in the supplied data. Retention is therefore always calculated from the corresponding retention flag, rather than inferred from the number of rounds. The next cell compares zero-round shares across the groups.

In [12]:
# Share of zero-round players in each group (%)
(df["sum_gamerounds"] == 0).groupby(df["version"]).mean().mul(100).round(2)

version
gate_30    4.33
gate_40    4.52
Name: sum_gamerounds, dtype: float64

 **Step 1.4 - Validate Retention Logic & Consistency**

In [13]:

# Check retention column data types
df[["retention_1", "retention_7"]].dtypes


retention_1    bool
retention_7    bool
dtype: object

**Step 1.5 - Check Retention Combinations**

In [14]:

# Examine retention combinations
pd.crosstab(
    df["retention_1"],
    df["retention_7"]
)


retention_7,False,True
retention_1,,
False,46437,3599
True,26971,13182


**Insight:** 13,182 players returned on both Day 1 and Day 7, while 3,599 players (about 4%) returned on Day 7 but not on Day 1. This is not a data error: retention is measured on a specific day, so a player can skip Day 1 and come back on Day 7. Overall, about 44.5% of players were retained on Day 1 and about 18.6% on Day 7.

**Step 1.6 - Descriptive Statistics**

In [15]:

# Summary statistics for game rounds
df["sum_gamerounds"].describe()


count    90189.000000
mean        51.872457
std        195.050858
min          0.000000
25%          5.000000
50%         16.000000
75%         51.000000
max      49854.000000
Name: sum_gamerounds, dtype: float64

**Step 1.7 - Analyze Game Rounds Distribution**

In [16]:


plt.figure(figsize=(10, 5))

plt.hist(
    df["sum_gamerounds"],
    bins=50
)

plt.title("Distribution of Game Rounds")
plt.xlabel("Number of Game Rounds")
plt.ylabel("Number of Players")

plt.show()


/tmp/ipykernel_1037/2862865722.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:

# Focus on players with 500 rounds or fewer
plt.figure(figsize=(10, 5))

plt.hist(
    df.loc[df["sum_gamerounds"] <= 500, "sum_gamerounds"],
    bins=50
)

plt.title("Game Rounds Distribution (0–500)")
plt.xlabel("Number of Game Rounds")
plt.ylabel("Number of Players")

plt.show()


/tmp/ipykernel_1037/1321450207.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:

plt.figure(figsize=(10, 4))

sns.boxplot(
    data=df,
    x="sum_gamerounds"
)

plt.title("Boxplot of Game Rounds")
plt.xlabel("Number of Game Rounds")

plt.show()


/tmp/ipykernel_1037/1163671300.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
# Visualize Game Rounds Using a Boxplot

plt.figure(figsize=(10, 4))

sns.boxplot(
    data=df,
    x="sum_gamerounds"
)

plt.xlim(0, 200)

plt.title("Game Rounds Boxplot (Zoomed View)")
plt.xlabel("Number of Game Rounds")

plt.show()



/tmp/ipykernel_1037/1732495606.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Step 1.8 - Identify Potential Outliers**

In [20]:

# Calculate Q1 and Q3
Q1 = df["sum_gamerounds"].quantile(0.25)
Q3 = df["sum_gamerounds"].quantile(0.75)

# Calculate IQR
IQR = Q3 - Q1

# Calculate outlier fences
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Fence:", lower_fence)
print("Upper Fence:", upper_fence)


Q1: 5.0
Q3: 51.0
IQR: 46.0
Lower Fence: -64.0
Upper Fence: 120.0


In [21]:

# Identify potential outliers
outlier_mask = (
    (df["sum_gamerounds"] < lower_fence) |
    (df["sum_gamerounds"] > upper_fence)
)

# Count potential outliers
outlier_mask.sum()


np.int64(10177)

**Step 1.9 - Assess Outlier Impact**

In [22]:

# Compare statistics with and without potential outliers

rounds = df["sum_gamerounds"]

# Keep observations within IQR fences
rounds_without_outliers = rounds[~outlier_mask]

comparison = pd.DataFrame({
    "All Players": [
        rounds.mean(),
        rounds.median()
    ],
    "Without Potential Outliers": [
        rounds_without_outliers.mean(),
        rounds_without_outliers.median()
    ]
}, index=["Mean", "Median"])

comparison.round(2)


,All Players,Without Potential Outliers
Mean,51.87,23.98
Median,16.00,13.00


**Insight:** the IQR rule flags 10,177 players (about 11%) as potential outliers, and removing them drops the mean from 51.87 to 23.98 while the median barely moves (16 to 13). The data is strongly right-skewed: a long tail of highly engaged players. These are legitimate players, not errors, so they are **not removed**. Only one record is truly suspicious (49,854 rounds versus 2,961 for the next highest player). It is examined separately in Step 4.5, and the rounds comparison relies on medians and a rank-based test.

# **Step 2 - Exploratory Data Analysis (EDA)**

**Step 2.1 -  Compare Experiment Groups**

In [23]:
# Count players in each experiment group
group_counts = df["version"].value_counts()

# Calculate percentage of players in each group
group_percentages = df["version"].value_counts(normalize=True) * 100

# Combine counts and percentages
group_summary = pd.DataFrame({
    "Player Count": group_counts,
    "Percentage (%)": group_percentages
})

group_summary.round(2)

,Player Count,Percentage (%)
version,,
gate_40,45489,50.44
gate_30,44700,49.56


**Step 2.1.1 - Sample Ratio Mismatch (SRM) Check**

In a randomized experiment with a 50/50 split, group sizes should be close to equal. A chi-square goodness-of-fit test checks whether the observed split is consistent with random assignment.

In [24]:
# SRM check: observed group sizes vs an expected 50/50 split
observed = df["version"].value_counts().reindex(["gate_30", "gate_40"])
expected = [len(df) / 2] * 2

chi2_stat, p_srm = chisquare(f_obs=observed, f_exp=expected)

print("Observed group sizes:", observed.to_dict())
print(f"Chi-square = {chi2_stat:.4f}, p-value = {p_srm:.4f}")

Observed group sizes: {'gate_30': 44700, 'gate_40': 45489}
Chi-square = 6.9024, p-value = 0.0086


**Interpretation:** the observed split is 49.56% vs 50.44%, with chi-square ≈ 6.90 and p ≈ 0.0086 under an **assumed 50/50 allocation**. This flags a potential sample-ratio mismatch. Although the absolute deviation is small and some monitoring practices use stricter alert thresholds, the result should **not** be dismissed solely because the percentages look close. In a live product experiment, confirm the intended allocation and audit randomization, eligibility and event logging before trusting a rollout decision. The statistical comparison below is exploratory, conditional on those checks.

**Step 2.2 - Analyze Game Rounds by Experiment Group**

In [25]:
# Compare game rounds between experiment groups
rounds_summary = df.groupby("version")["sum_gamerounds"].agg(
    Player_Count="count",
    Mean="mean",
    Median="median",
    Q1=lambda x: x.quantile(0.25),
    Q3=lambda x: x.quantile(0.75),
    Max="max"
)

rounds_summary.round(2)

,Player_Count,Mean,Median,Q1,Q3,Max
version,,,,,,
gate_30,44700,52.46,17.0,5.0,50.0,49854
gate_40,45489,51.30,16.0,5.0,52.0,2640


**Step 2.3 - Visualize Game Rounds Distribution by Experiment Group**

In [26]:


plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df,
    x="version",
    y="sum_gamerounds"
)

plt.yscale("log")
plt.title("Game Rounds Distribution by Experiment Group")
plt.xlabel("Experiment Group")
plt.ylabel("Game Rounds (Log Scale)")

plt.tight_layout()
plt.show()

/tmp/ipykernel_1037/2202239167.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Step 2.4 - Analyze Retention by Experiment Group**

In [27]:
# Calculate retention rates for each experiment group
retention_summary = df.groupby("version").agg(
    Player_Count=("userid", "count"),
    Day_1_Retained=("retention_1", "mean"),
    Day_7_Retained=("retention_7", "mean")
)

# Convert retention rates to percentages
retention_summary[
    ["Day_1_Retained", "Day_7_Retained"]
] *= 100

retention_summary.round(2)

,Player_Count,Day_1_Retained,Day_7_Retained
version,,,
gate_30,44700,44.82,19.02
gate_40,45489,44.23,18.20


**Step 2.5 - Visualize Retention Rates by Experiment Group**

In [28]:


# Prepare retention data for visualization
retention_plot = retention_summary[
    ["Day_1_Retained", "Day_7_Retained"]
]

# Plot retention rates
retention_plot.plot(
    kind="bar",
    figsize=(9, 5)
)

plt.title("Retention Rates by Experiment Group")
plt.xlabel("Experiment Group")
plt.ylabel("Retention Rate (%)")
plt.xticks(rotation=0)
plt.legend(
    ["Day 1 Retention", "Day 7 Retention"],
    title="Metric"
)
plt.tight_layout()
plt.show()

/tmp/ipykernel_1037/1160571663.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# **Step 3 - Statistical Testing**

**Step 3.1 - Choose the Statistical Test**
Selected Test: Two-Proportion Z-Test
Rationale:

*   The experiment contains two independent groups: gate_30 and gate_40.
*   Both retention metrics are binary outcomes (True or False).
*   The objective is to compare retention proportions between the two groups.
*   A Two-Proportion Z-Test is appropriate for comparing two independent proportions when its assumptions are satisfied.

Testing Plan:


*   Conduct a separate test for Day 1 Retention and Day 7 Retention.
*   Evaluate p-values alongside effect sizes and confidence intervals.
*   Account for multiple testing when interpreting the two retention results.
Important: Observed differences in retention rates do not, by themselves, establish statistical significance or prove a causal effect.










**Step 3.2 - Prepare the Data for Hypothesis Testing**

In [29]:

retention_counts = df.groupby("version").agg(
    Total_Players=("userid", "count"),
    Day_1_Retained=("retention_1", "sum"),
    Day_7_Retained=("retention_7", "sum")
)

retention_counts["Day_1_Rate"] = (
    retention_counts["Day_1_Retained"]
    / retention_counts["Total_Players"]
)

retention_counts["Day_7_Rate"] = (
    retention_counts["Day_7_Retained"]
    / retention_counts["Total_Players"]
)

retention_counts


,Total_Players,Day_1_Retained,Day_7_Retained,Day_1_Rate,Day_7_Rate
version,,,,,
gate_30,44700,20034,8502,0.448188,0.190201
gate_40,45489,20119,8279,0.442283,0.182000


**Step 3.3 - Performing the Two-Proportion Z-Test**

In [30]:
# Number of players retained on Day 1
count_day1 = [
    retention_counts.loc["gate_30", "Day_1_Retained"],
    retention_counts.loc["gate_40", "Day_1_Retained"]
]

# Total players in each group
nobs = [
    retention_counts.loc["gate_30", "Total_Players"],
    retention_counts.loc["gate_40", "Total_Players"]
]

# Day 1 test
z_day1, p_day1 = proportions_ztest(
    count=count_day1,
    nobs=nobs,
    alternative="two-sided"
)

# Number of players retained on Day 7
count_day7 = [
    retention_counts.loc["gate_30", "Day_7_Retained"],
    retention_counts.loc["gate_40", "Day_7_Retained"]
]

# Day 7 test
z_day7, p_day7 = proportions_ztest(
    count=count_day7,
    nobs=nobs,
    alternative="two-sided"
)

print(f"Day 1: Z = {z_day1:.4f}, P-value = {p_day1:.6f}")
print(f"Day 7: Z = {z_day7:.4f}, P-value = {p_day7:.6f}")

Day 1: Z = 1.7841, P-value = 0.074410
Day 7: Z = 3.1644, P-value = 0.001554


**Step 3.4 - Multiple Testing Correction**

In [31]:

# Multiple Testing Correction: Bonferroni

alpha = 0.05
number_of_tests = 2

adjusted_alpha = alpha / number_of_tests

print(f"Original alpha: {alpha}")
print(f"Adjusted alpha: {adjusted_alpha}")
print(f"Day 1 significant: {p_day1 < adjusted_alpha}")
print(f"Day 7 significant: {p_day7 < adjusted_alpha}")


Original alpha: 0.05
Adjusted alpha: 0.025
Day 1 significant: False
Day 7 significant: True


# Statistical Testing Summary

A two-proportion Z-test was conducted to compare Day 1 and Day 7 retention between Gate 30 and Gate 40. Bonferroni correction was applied to account for the two hypothesis tests, resulting in an adjusted significance level of 0.025.

- **Day 1 Retention:** Gate 30 had a higher observed retention rate (44.82%) than Gate 40 (44.23%), a difference of 0.59 percentage points. However, the difference was not statistically significant (p = 0.074410).

- **Day 7 Retention:** Gate 30 had a higher observed retention rate (19.02%) than Gate 40 (18.20%), a difference of 0.82 percentage points. This difference remained statistically significant after Bonferroni correction (p = 0.001554).

**Conclusion:** The observed results favor Gate 30 for Day 7 retention, while the evidence is insufficient to establish a difference in Day 1 retention. Further assessment of effect size, confidence intervals, and practical significance is needed before making a final product recommendation.

**Step 3.5 - Power Analysis: Minimum Detectable Effect (MDE)**

Power analysis shows the smallest difference this experiment could reliably detect (80% power, alpha = 0.025, using the actual group sizes). It helps interpret a non-significant result: "no significant difference" is not the same as "no effect".

In [32]:
power_analysis = NormalIndPower()

n_gate30 = retention_counts.loc["gate_30", "Total_Players"]
n_gate40 = retention_counts.loc["gate_40", "Total_Players"]
size_ratio = n_gate40 / n_gate30

def minimum_detectable_effect(baseline_rate, alpha=0.025, power=0.80):
    """Smallest detectable difference (in percentage points) for a given baseline rate."""
    h = power_analysis.solve_power(
        effect_size=None,
        nobs1=n_gate30,
        alpha=alpha,
        power=power,
        ratio=size_ratio,
        alternative="two-sided"
    )
    alt_rate = np.sin(np.arcsin(np.sqrt(baseline_rate)) + h / 2) ** 2
    return (alt_rate - baseline_rate) * 100

for label, col in [("Day 1", "Day_1_Rate"), ("Day 7", "Day_7_Rate")]:
    baseline = retention_counts.loc["gate_40", col]
    mde = minimum_detectable_effect(baseline)
    print(f"{label}: baseline = {baseline:.2%}, MDE = {mde:.2f} percentage points")

Day 1: baseline = 44.23%, MDE = 1.02 percentage points
Day 7: baseline = 18.20%, MDE = 0.80 percentage points


**Interpretation (approximate, retrospective):** at 80% power under the observed baseline rates, this experiment has a calculated MDE of about 0.8 percentage points for Day 7 and about 1.0 percentage point for Day 1. The observed Day 1 difference (0.59 pp) is smaller than the 80%-power MDE, helping contextualize the inconclusive result. This is a retrospective sensitivity calculation, **not** proof of the absence of a smaller effect or a substitute for pre-experiment power planning.

# **Step 4**

**Step 4.1: Review Statistical Testing Results**

*   Day 1: insufficient statistical evidence of a difference between the groups.
*   Day 7: statistically significant difference in favor of Gate 30.

**Step 4.2: Calculate Effect Size**

In [33]:


# Day 1 Retention
day1_gate30 = retention_counts.loc["gate_30", "Day_1_Rate"]
day1_gate40 = retention_counts.loc["gate_40", "Day_1_Rate"]

day1_diff_pp = (day1_gate30 - day1_gate40) * 100
day1_relative_diff = ((day1_gate30 - day1_gate40) / day1_gate40) * 100

# Day 7 Retention
day7_gate30 = retention_counts.loc["gate_30", "Day_7_Rate"]
day7_gate40 = retention_counts.loc["gate_40", "Day_7_Rate"]

day7_diff_pp = (day7_gate30 - day7_gate40) * 100
day7_relative_diff = ((day7_gate30 - day7_gate40) / day7_gate40) * 100

print("Day 1 Retention")
print(f"Absolute Difference: {day1_diff_pp:.2f} percentage points")
print(f"Relative Difference: {day1_relative_diff:.2f}%")

print("\nDay 7 Retention")
print(f"Absolute Difference: {day7_diff_pp:.2f} percentage points")
print(f"Relative Difference: {day7_relative_diff:.2f}%")


Day 1 Retention
Absolute Difference: 0.59 percentage points
Relative Difference: 1.34%

Day 7 Retention
Absolute Difference: 0.82 percentage points
Relative Difference: 4.51%


**Step 4.3 — Confidence Intervals**

In [34]:
# 97.5% confidence intervals (Bonferroni-adjusted: 0.05 / 2 tests = 0.025)
alpha_ci = 0.025

def retention_diff_ci(retained_col):
    """CI for the difference in retention rate: Gate 30 minus Gate 40."""
    return confint_proportions_2indep(
        count1=int(retention_counts.loc["gate_30", retained_col]),
        nobs1=int(retention_counts.loc["gate_30", "Total_Players"]),
        count2=int(retention_counts.loc["gate_40", retained_col]),
        nobs2=int(retention_counts.loc["gate_40", "Total_Players"]),
        method="newcomb",
        compare="diff",
        alpha=alpha_ci
    )

day1_ci = retention_diff_ci("Day_1_Retained")
day7_ci = retention_diff_ci("Day_7_Retained")

print("Day 1 Confidence Interval (97.5%):")
print(f"Lower Bound: {day1_ci[0] * 100:.2f}%")
print(f"Upper Bound: {day1_ci[1] * 100:.2f}%")

print("\nDay 7 Confidence Interval (97.5%):")
print(f"Lower Bound: {day7_ci[0] * 100:.2f}%")
print(f"Upper Bound: {day7_ci[1] * 100:.2f}%")

Day 1 Confidence Interval (97.5%):
Lower Bound: -0.15%
Upper Bound: 1.33%

Day 7 Confidence Interval (97.5%):
Lower Bound: 0.24%
Upper Bound: 1.40%


**Interpretation:** the Day 7 interval (0.24 to 1.40 percentage points) does not include zero, in line with the adjusted significant result. Under the model and its assumptions, this interval describes the uncertainty around the Gate 30 minus Gate 40 retention difference. The Day 1 interval (-0.15 to 1.33 pp) includes zero, so a Day 1 difference remains inconclusive.

**Note: two different questions**

- *Statistical significance*: do we have statistical evidence that a difference exists?
- *Practical significance*: is the size of the difference large enough to justify changing the product decision?

**Step 4.4 - Statistical vs Practical Significance**

The Day 7 difference is small in absolute terms (0.82 percentage points, about 4.51% relative to Gate 40). Small effects can still matter at scale, so the next cell translates the result into a number of players for an illustrative cohort of 100,000 new installs.

In [35]:
# Practical significance: impact on an illustrative cohort of new players
cohort_size = 100_000

scenarios = {
    "Observed difference": day7_diff_pp / 100,
    "CI lower bound": day7_ci[0],
    "CI upper bound": day7_ci[1],
}

for label, diff in scenarios.items():
    print(f"{label}: about {diff * cohort_size:,.0f} additional Day 7 retained players per {cohort_size:,} new players")

Observed difference: about 820 additional Day 7 retained players per 100,000 new players
CI lower bound: about 239 additional Day 7 retained players per 100,000 new players
CI upper bound: about 1,401 additional Day 7 retained players per 100,000 new players


**Interpretation:** for every 100,000 new players, Gate 30 is expected to keep roughly 820 more players active at Day 7, with a plausible range of about 240 to 1,400. Whether that justifies the decision depends on the value of a retained player (revenue, ads, lifetime value), which is not available in this dataset.

**Step 4.5: Analyze Game Rounds & Outliers**

In [36]:

# Compare game rounds between experiment groups
rounds_comparison = df.groupby("version")["sum_gamerounds"].agg(
    Player_Count="count",
    Mean="mean",
    Median="median",
    Q1=lambda x: x.quantile(0.25),
    Q3=lambda x: x.quantile(0.75),
    Max="max"
)

rounds_comparison.round(2)


,Player_Count,Mean,Median,Q1,Q3,Max
version,,,,,,
gate_30,44700,52.46,17.0,5.0,50.0,49854
gate_40,45489,51.30,16.0,5.0,52.0,2640


In [37]:

# Inspect the 10 players with the highest game rounds

df[["version", "sum_gamerounds"]].sort_values(
    by="sum_gamerounds",
    ascending=False
).head(10)


,version,sum_gamerounds
57702,gate_30,49854
7912,gate_30,2961
29417,gate_40,2640
43671,gate_30,2438
48188,gate_40,2294
46344,gate_30,2251
87007,gate_30,2156
36933,gate_40,2124
88328,gate_40,2063
6536,gate_40,2015


In [38]:

# Compare Gate 30 mean before and after excluding the extreme value

gate30_rounds = df.loc[
    df["version"] == "gate_30", "sum_gamerounds"
]

mean_original = gate30_rounds.mean()
mean_without_extreme = gate30_rounds[
    gate30_rounds != gate30_rounds.max()
].mean()

print("Original mean:", round(mean_original, 2))
print("Mean without maximum value:", round(mean_without_extreme, 2))


Original mean: 52.46
Mean without maximum value: 51.34


In [39]:

# Sensitivity analysis: compare rounds without the extreme value

df_sensitivity = df[
    ~((df["version"] == "gate_30") &
      (df["sum_gamerounds"] == 49854))
]

df_sensitivity.groupby("version")["sum_gamerounds"].agg(
    Mean="mean",
    Median="median",
    Player_Count="count"
).round(2)


,Mean,Median,Player_Count
version,,,
gate_30,51.34,17.0,44699
gate_40,51.30,16.0,45489


**Step 4.6 - Apply Mann–Whitney U Test on Game Rounds (Secondary Metric)**

In [40]:
# Secondary metric: Mann-Whitney U test on game rounds
gate30 = df.loc[df["version"] == "gate_30", "sum_gamerounds"]
gate40 = df.loc[df["version"] == "gate_40", "sum_gamerounds"]

u_statistic, p_value = mannwhitneyu(
    gate30,
    gate40,
    alternative="two-sided"
)

# Secondary metric: interpreted descriptively, not part of the Bonferroni correction
alpha_secondary = 0.05

print("U-statistic:", u_statistic)
print("P-value:", round(p_value, 4))

if p_value < alpha_secondary:
    print("Result: Statistically significant difference")
elif p_value < 0.10:
    print("Result: Borderline - not significant at the 5% level; evidence is inconclusive")
else:
    print("Result: No statistically significant difference")

U-statistic: 1024331250.5
P-value: 0.0502
Result: Borderline - not significant at the 5% level; evidence is inconclusive


**Interpretation:** the Mann–Whitney U test compares distributions of game rounds (useful for the skewed data, though its p-value does not directly test a difference in medians). With p ≈ 0.0502, the result is **not significant at 5%**; this does not demonstrate equivalence. Descriptively, medians are 17 vs 16 rounds. The Gate 30 mean moves from 52.46 to 51.34 if the single extreme observation is excluded, versus 51.30 in Gate 40. All players remain in the main test; exclusion is a sensitivity check only. Game rounds are a secondary, exploratory metric.

**Step 4.7 - Final Business Recommendation**

**Recommendation: do not move the gate to level 40. Keep Gate 30.**

**Evidence**
- **Day 7 retention:** 19.02% (Gate 30) vs 18.20% (Gate 40), a difference of 0.82 percentage points (97.5% CI: 0.24 to 1.40; p = 0.0016, significant after Bonferroni correction).
- **Day 1 retention:** 0.59 percentage points, not significant (p = 0.074, CI includes 0). The experiment could only detect differences of about 1 point or more at Day 1, so this result is inconclusive.
- **Game rounds:** medians are 17 vs 16, with an inconclusive Mann–Whitney U result (p ≈ 0.0502); these data do not establish improved engagement under Gate 40.
- **Practical impact:** about 820 additional Day 7 retained players per 100,000 new players (plausible range 240 to 1,400).

**Limitations**
- Mild sample ratio mismatch (p ≈ 0.0086): assignment and logging should be verified.
- Revenue, ad impressions and other monetization metrics are not in the data. Gates may influence monetization, so retention alone does not settle the decision.
- Only Day 1 and Day 7 retention are measured, and the result comes from a single experiment.

**Next steps:** first verify the intended allocation and logging due to the SRM flag; then evaluate revenue and lifetime value per group, and consider longer retention windows before a final rollout decision.

# **Step 5 - Visualization**

**Step 5.1 - Prepare the Results Tables for the Charts**

The tables below are built directly from the results computed above, so every chart uses the same numbers as the analysis.

In [41]:
# 1. Player Distribution
player_distribution = (
    df["version"]
    .value_counts()
    .reindex(["gate_30", "gate_40"])
    .rename_axis("Version")
    .reset_index(name="Player_Count")
)
player_distribution["Percentage"] = (
    player_distribution["Player_Count"] / len(df) * 100
).round(2)

# 2. Retention Comparison (computed from the results above)
retention_data = pd.DataFrame({
    "Metric": ["Day 1 Retention", "Day 7 Retention"],
    "Gate 30": [retention_counts.loc["gate_30", "Day_1_Rate"] * 100,
                retention_counts.loc["gate_30", "Day_7_Rate"] * 100],
    "Gate 40": [retention_counts.loc["gate_40", "Day_1_Rate"] * 100,
                retention_counts.loc["gate_40", "Day_7_Rate"] * 100],
}).round(2)
retention_data["Difference_pp"] = (
    retention_data["Gate 30"] - retention_data["Gate 40"]
).round(2)

# 3. Statistical Results
statistical_results = pd.DataFrame({
    "Metric": ["Day 1 Retention", "Day 7 Retention", "Game Rounds"],
    "Metric_Role": ["Primary", "Primary", "Secondary"],
    "Test": ["Z-test", "Z-test", "Mann–Whitney U"],
    "Statistic": [z_day1, z_day7, u_statistic],
    "P_Value": [p_day1, p_day7, p_value],
    "Alpha": [adjusted_alpha, adjusted_alpha, alpha_secondary],
    "Significant": [p_day1 < adjusted_alpha, p_day7 < adjusted_alpha, p_value < alpha_secondary],
})

# 4. Confidence Intervals (97.5%, retention differences)
confidence_intervals = pd.DataFrame({
    "Metric": ["Day 1 Retention Difference", "Day 7 Retention Difference"],
    "Difference_pp": [day1_diff_pp, day7_diff_pp],
    "CI_Lower_pp": [day1_ci[0] * 100, day7_ci[0] * 100],
    "CI_Upper_pp": [day1_ci[1] * 100, day7_ci[1] * 100],
}).round(2)

# 5. Game Rounds Summary by Version
game_rounds_summary = (
    df.groupby("version")["sum_gamerounds"]
    .agg(
        Player_Count="count",
        Mean="mean",
        Median="median",
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75),
        Max="max",
    )
    .round(2)
    .reset_index()
)

# 6. Outlier Sensitivity Summary
outlier_sensitivity = pd.DataFrame({
    "Scenario": [
        "Gate 30 - Original",
        "Gate 30 - Without Maximum Outlier",
        "Gate 40 - Original",
    ],
    "Mean_Game_Rounds": [
        mean_original,
        mean_without_extreme,
        rounds_comparison.loc["gate_40", "Mean"],
    ],
}).round(2)

# Display all prepared tables
for title, table in [
    ("1. PLAYER DISTRIBUTION", player_distribution),
    ("2. RETENTION COMPARISON", retention_data),
    ("3. STATISTICAL RESULTS", statistical_results),
    ("4. CONFIDENCE INTERVALS", confidence_intervals),
    ("5. GAME ROUNDS SUMMARY", game_rounds_summary),
    ("6. OUTLIER SENSITIVITY", outlier_sensitivity),
]:
    print(title)
    display(table)

1. PLAYER DISTRIBUTION

,Version,Player_Count,Percentage
0,gate_30,44700,49.56
1,gate_40,45489,50.44


2. RETENTION COMPARISON


,Metric,Gate 30,Gate 40,Difference_pp
0,Day 1 Retention,44.82,44.23,0.59
1,Day 7 Retention,19.02,18.20,0.82


3. STATISTICAL RESULTS


,Metric,Metric_Role,Test,Statistic,P_Value,Alpha,Significant
0,Day 1 Retention,Primary,Z-test,1.784086e+00,0.074410,0.025,False
1,Day 7 Retention,Primary,Z-test,3.164359e+00,0.001554,0.025,True
2,Game Rounds,Secondary,Mann–Whitney U,1.024331e+09,0.050209,0.050,False


... (2 outputs left out)

5. GAME ROUNDS SUMMARY


,version,Player_Count,Mean,Median,Q1,Q3,Max
0,gate_30,44700,52.46,17.0,5.0,50.0,49854
1,gate_40,45489,51.30,16.0,5.0,52.0,2640


6. OUTLIER SENSITIVITY


,Scenario,Mean_Game_Rounds
0,Gate 30 - Original,52.46
1,Gate 30 - Without Maximum Outlier,51.34
2,Gate 40 - Original,51.30


**Step 5.1.1 - Export Reproducible Summary Tables**

The following cell writes the six result tables used in the project. All tables are generated from the included raw CSV; no manual spreadsheet calculations are required.

In [42]:
# Export the six analysis tables so README results are traceable to the notebook.
from pathlib import Path
output_dir = Path("results")
output_dir.mkdir(parents=True, exist_ok=True)

result_tables = {
    "player_distribution.csv": player_distribution,
    "retention_comparison.csv": retention_data,
    "statistical_results.csv": statistical_results,
    "confidence_intervals.csv": confidence_intervals,
    "game_rounds_summary.csv": game_rounds_summary,
    "outlier_sensitivity.csv": outlier_sensitivity,
}
for filename, table in result_tables.items():
    table.to_csv(output_dir / filename, index=False)
print(f"Saved {len(result_tables)} summary tables to {output_dir}/")

Saved 6 summary tables to results/


**Step 5.2 - Core Charts: Retention and Confidence Intervals**

All figures are saved in a `figures/` folder so they can be reused in the project README. Takeaway titles state the finding, not just the metric.

In [43]:
import os
os.makedirs("figures", exist_ok=True)

In [44]:
# Chart 1: confidence intervals of the retention difference (Gate 30 - Gate 40)
fig, ax = plt.subplots(figsize=(9, 3.8))

results = [("Day 1", day1_diff_pp, day1_ci), ("Day 7", day7_diff_pp, day7_ci)]

for i, (label, diff, ci) in enumerate(results):
    lo, hi = ci[0] * 100, ci[1] * 100
    color = "#2E9E5B" if lo > 0 else "#E8B400"
    ax.plot([lo, hi], [i, i], lw=6, color=color, solid_capstyle="round")
    ax.plot(diff, i, "o", color="black", markersize=8)
    ax.text(diff, i + 0.2, f"{diff:+.2f} pp  [{lo:.2f}, {hi:.2f}]", ha="center")

ax.axvline(0, color="red", linestyle="--", linewidth=1.5)
ax.set_yticks([0, 1])
ax.set_yticklabels(["Day 1", "Day 7"])
ax.set_ylim(-0.5, 1.7)
ax.set_xlabel("Difference in retention, Gate 30 minus Gate 40 (percentage points)")
ax.set_title("Gate 30 retains more players at Day 7; Day 1 is inconclusive (97.5% CI)")
plt.tight_layout()
plt.savefig("figures/01_retention_difference_ci.png", dpi=200, bbox_inches="tight")
plt.show()

/tmp/ipykernel_1037/3113905454.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [45]:
# Chart 2: retention rates with value labels
plot_df = retention_summary[["Day_1_Retained", "Day_7_Retained"]].T
plot_df.index = ["Day 1", "Day 7"]
plot_df.columns = ["Gate 30", "Gate 40"]

ax = plot_df.plot(kind="bar", figsize=(8, 5), color=["#12239E", "#118DFF"])
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f%%", padding=3)

ax.set_ylim(0, 55)
ax.set_ylabel("Retention Rate (%)")
ax.set_xlabel("")
ax.set_title("Retention Rate by Gate Version")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("figures/02_retention_by_gate.png", dpi=200, bbox_inches="tight")
plt.show()

/tmp/ipykernel_1037/2968318203.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**How to read these charts**

- **Retention by gate:** Gate 30 is slightly higher at both Day 1 and Day 7. The gap is visible but small, which is why the statistical tests matter.
- **Confidence intervals:** the Day 7 interval stays entirely to the right of zero (Gate 30 is better), while the Day 1 interval crosses zero (inconclusive).

**Step 5.3 - Game Rounds: Typical Activity and the Outlier Effect**

In [46]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
order = ["gate_30", "gate_40"]
colors = ["#12239E", "#118DFF"]

# Left: distribution of rounds (capped at 200 rounds for readability)
capped = df[df["sum_gamerounds"] <= 200]
box_data = [capped.loc[capped["version"] == v, "sum_gamerounds"] for v in order]
bp = axes[0].boxplot(box_data, patch_artist=True, showfliers=False,
                     medianprops=dict(color="white", linewidth=2))
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
axes[0].set_xticks([1, 2])
axes[0].set_xticklabels(["Gate 30", "Gate 40"])
medians = df.groupby("version")["sum_gamerounds"].median()
axes[0].set_xlabel(f"Median rounds: Gate 30 = {medians['gate_30']:.0f}, Gate 40 = {medians['gate_40']:.0f}")
axes[0].set_ylabel("Game rounds (first 14 days)")
axes[0].set_title("Typical player activity is almost identical\n(players with 200 rounds or fewer)")

# Right: effect of the single extreme player on the mean
labels = outlier_sensitivity["Scenario"].str.replace(" - ", "\n", regex=False)
bars = axes[1].bar(range(len(outlier_sensitivity)), outlier_sensitivity["Mean_Game_Rounds"],
                   color=["#12239E", "#7C86C9", "#118DFF"])
axes[1].bar_label(bars, fmt="%.2f", padding=3)
axes[1].set_xticks(range(len(outlier_sensitivity)))
axes[1].set_xticklabels(labels)
axes[1].set_ylim(0, outlier_sensitivity["Mean_Game_Rounds"].max() * 1.2)
axes[1].set_ylabel("Mean game rounds")
axes[1].set_title("One extreme player (49,854 rounds) inflates Gate 30's mean")

for a in axes:
    a.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("figures/03_game_rounds.png", dpi=200, bbox_inches="tight")
plt.show()

/tmp/ipykernel_1037/3272734068.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation:** the medians are 17 and 16 rounds. The mean difference is highly sensitive to the single extreme value: Gate 30 falls from 52.46 to 51.34 rounds without that observation, versus 51.30 for Gate 40. This motivates reporting robust descriptive statistics alongside a rank-based test; it does not prove the distributions are equivalent. The boxplot on the left is limited to players with 200 rounds or fewer **for visual readability only**.

**Step 5.4 - Practical Impact of the Day 7 Difference**

In [47]:
cohort_size = 100_000  # illustrative cohort of new installs

low = day7_ci[0] * cohort_size
mid = day7_diff_pp / 100 * cohort_size
high = day7_ci[1] * cohort_size

fig, ax = plt.subplots(figsize=(9, 2.8))
ax.barh([0], [high - low], left=[low], height=0.35, color="#2E9E5B", alpha=0.35)
ax.plot(mid, 0, "o", color="black", markersize=9)
ax.text(mid, 0.33, f"about {mid:,.0f}  (range {low:,.0f} to {high:,.0f})", ha="center")

ax.set_xlim(0, high * 1.25)
ax.set_ylim(-0.5, 0.8)
ax.set_yticks([])
ax.set_xlabel(f"Extra players retained at Day 7 per {cohort_size:,} new installs (illustrative)")
ax.set_title("Gate 30 is expected to keep roughly 820 more players at Day 7 per 100K installs")
ax.spines[["top", "right", "left"]].set_visible(False)

plt.tight_layout()
plt.savefig("figures/04_practical_impact.png", dpi=200, bbox_inches="tight")
plt.show()

/tmp/ipykernel_1037/3585238176.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation:** the observed Day 7 difference favors Gate 30, but it is modest and uncertain. The projected number of additional retained players is an illustration for a hypothetical 100,000-player cohort, **not an observed outcome**. Monetization data are absent, and the SRM signal needs follow-up before treating this as a production rollout decision.

**Step 5.5 - One-Page Executive Summary**

A single figure that brings the key results together (for the README or a quick stakeholder update). All numbers are taken from the analysis above.

In [48]:
from matplotlib.patches import FancyBboxPatch, Rectangle

BG, CARD, DARK = "#EEF0F3", "#FFFFFF", "#1F2A44"
NAVY, BLUE, GREEN, YELLOW, RED, GRAY = "#12239E", "#118DFF", "#2E9E5B", "#E8B400", "#D64550", "#6B7280"

def verdict(p, alpha):
    if p < alpha:
        return "Significant", GREEN
    if p < 0.10:
        return "Inconclusive", YELLOW
    return "Not significant", GRAY

fig = plt.figure(figsize=(16, 9), dpi=130, facecolor=BG)
ax = fig.add_axes([0, 0, 1, 1])
ax.set_xlim(0, 16); ax.set_ylim(0, 9); ax.axis("off")

def card(x, y, w, h):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0,rounding_size=0.08",
                                fc=CARD, ec="#D9DDE3", lw=1))

n30 = int(retention_counts.loc["gate_30", "Total_Players"])
n40 = int(retention_counts.loc["gate_40", "Total_Players"])
v1, c1 = verdict(p_day1, adjusted_alpha)
v7, c7 = verdict(p_day7, adjusted_alpha)
vr, cr = verdict(p_value, alpha_secondary)

# Header
ax.add_patch(Rectangle((0, 8.25), 16, 0.75, fc=DARK, ec="none"))
ax.text(0.4, 8.62, "A/B Test: Impact of Gate Placement on Player Retention",
        color="white", fontsize=17, fontweight="bold", va="center")
ax.text(15.6, 8.62, f"Cookie Cats  |  {len(df):,} players", color="#C9D1E3", fontsize=11, va="center", ha="right")

# KPI cards
kpis = [
    ("Total Players", f"{len(df):,}", f"Gate 30: {n30:,}  |  Gate 40: {n40:,}", DARK),
    ("Day 1 Difference", f"{day1_diff_pp:+.2f} pp", f"{v1} (CI {day1_ci[0]*100:.2f} to {day1_ci[1]*100:.2f})", c1),
    ("Day 7 Difference", f"{day7_diff_pp:+.2f} pp", f"97.5% CI: {day7_ci[0]*100:.2f} to {day7_ci[1]*100:.2f}", c7),
    ("Day 7 P-value", f"{p_day7:.4f}", f"{v7} (alpha = {adjusted_alpha})", BLUE),
]
for i, (title, value, sub, color) in enumerate(kpis):
    x = 0.35 + i * 3.9
    card(x, 6.85, 3.65, 1.2)
    ax.add_patch(Rectangle((x, 6.85), 0.09, 1.2, fc=color, ec="none"))
    ax.text(x + 0.3, 7.8, title, fontsize=10.5, color=GRAY, va="center")
    ax.text(x + 0.3, 7.35, value, fontsize=22, fontweight="bold", color=DARK, va="center")
    ax.text(x + 0.3, 7.0, sub, fontsize=9, color=GRAY, va="center")

# Panel 1: retention bars
card(0.35, 2.35, 5.2, 4.25)
ax.text(0.6, 6.35, "Retention Rate by Gate Version", fontsize=12, fontweight="bold", color=DARK, va="center")
a1 = fig.add_axes([0.065, 0.29, 0.26, 0.40], facecolor=CARD)
xx = np.arange(2); w = 0.36
b1 = a1.bar(xx - w/2, retention_data["Gate 30"], w, color=NAVY, label="Gate 30")
b2 = a1.bar(xx + w/2, retention_data["Gate 40"], w, color=BLUE, label="Gate 40")
for b in list(b1) + list(b2):
    a1.text(b.get_x() + b.get_width()/2, b.get_height() + 0.8, f"{b.get_height():.2f}%", ha="center", fontsize=9, color=DARK)
a1.set_xticks(xx); a1.set_xticklabels(["Day 1", "Day 7"], fontsize=10)
a1.set_ylim(0, retention_data[["Gate 30", "Gate 40"]].values.max() * 1.18); a1.set_yticks([])
a1.spines[["top", "right", "left"]].set_visible(False)
a1.legend(frameon=False, fontsize=9, loc="upper right")

# Panel 2: confidence intervals
card(5.75, 2.35, 5.2, 4.25)
ax.text(6.0, 6.35, "Difference in Retention (Gate 30 - Gate 40)", fontsize=12, fontweight="bold", color=DARK, va="center")
ax.text(6.0, 6.05, "97.5% CI, Bonferroni-adjusted", fontsize=9, color=GRAY, va="center")
a2 = fig.add_axes([0.385, 0.29, 0.26, 0.35], facecolor=CARD)
for i, (label, diff, ci) in enumerate([("Day 1", day1_diff_pp, day1_ci), ("Day 7", day7_diff_pp, day7_ci)]):
    lo, hi = ci[0] * 100, ci[1] * 100
    a2.plot([lo, hi], [i, i], color=GREEN if lo > 0 else YELLOW, lw=5, solid_capstyle="round")
    a2.plot(diff, i, "o", color=DARK, ms=9)
    a2.text(diff, i + 0.25, f"{diff:+.2f}  [{lo:.2f}, {hi:.2f}]", ha="center", fontsize=9, color=DARK)
a2.axvline(0, color=RED, ls="--", lw=1.5)
a2.text(0, -0.62, "no difference", color=RED, fontsize=8.5, ha="center")
a2.set_yticks([0, 1]); a2.set_yticklabels(["Day 1", "Day 7"], fontsize=10)
a2.set_ylim(-0.8, 1.6)
a2.set_xlabel("percentage points", fontsize=9, color=GRAY)
a2.spines[["top", "right", "left"]].set_visible(False)
a2.tick_params(axis="y", length=0)

# Panel 3: test results table
card(11.15, 2.35, 4.5, 4.25)
ax.text(11.4, 6.35, "Statistical Test Results", fontsize=12, fontweight="bold", color=DARK, va="center")
rows = [("Metric", "Test", "P-value", "Result", DARK),
        ("Day 1 Retention", "Z-test", f"{p_day1:.4f}", v1, c1),
        ("Day 7 Retention", "Z-test", f"{p_day7:.4f}", v7, c7),
        ("Game Rounds", "Mann-Whitney", f"{p_value:.4f}", vr, cr)]
cols = [11.4, 12.85, 13.95, 14.6]
for r, row in enumerate(rows):
    y = 5.85 - r * 0.62
    if r == 0:
        ax.add_patch(Rectangle((11.3, y - 0.28), 4.25, 0.56, fc="#F1F3F7", ec="none"))
    for c in range(4):
        ax.text(cols[c], y, row[c], fontsize=9, va="center",
                fontweight="bold" if (r == 0 or c == 3) else "normal",
                color=row[4] if c == 3 else DARK)
ax.text(11.4, 3.15, f"Alpha: {adjusted_alpha} (retention)  |  {alpha_secondary} (game rounds)", fontsize=8.5, color=GRAY, va="center")
ax.text(11.4, 2.85, f"Outlier check: Gate 30 mean {mean_original:.2f} -> {mean_without_extreme:.2f}\nafter removing the single extreme player",
        fontsize=8.5, color=GRAY, va="top")

# Recommendation
card(0.35, 0.3, 15.3, 1.8)
ax.add_patch(Rectangle((0.35, 0.3), 0.12, 1.8, fc=GREEN, ec="none"))
ax.text(0.75, 1.75, "Recommendation", fontsize=13, fontweight="bold", color=DARK, va="center")
ax.text(0.75, 1.2,
        f"Keep the gate at level 30. Moving it to level 40 did not improve retention, and Day 7 retention was\n"
        f"{day7_diff_pp:.2f} pp lower with Gate 40. Before a final decision, validate the impact on revenue and player lifetime value.",
        fontsize=11, color=DARK, va="center", linespacing=1.6)
ax.text(0.75, 0.5,
        f"Limitations: mild sample ratio mismatch (p = {p_srm:.4f}); revenue not measured; Day 1 and game rounds results are inconclusive.",
        fontsize=9, color=GRAY, va="center")

plt.savefig("figures/05_executive_summary.png", dpi=150, facecolor=BG)
plt.show()

/tmp/ipykernel_1037/395458169.py:113: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Project outputs:** the `figures/` directory contains the five charts used in this notebook. The `results/` directory holds six CSV summary tables generated directly from the analysis. `figures/05_executive_summary.png` is the cover image in `README.md`.